# SUNSHIELD — Notebook 2: LSTM normal-behaviour model

**Place in Dr. Mai's workflow:** `Training (LSTM) → gives predicted line` and, for the test,
`LSTM (trained) → gives predicted line`.

The LSTM answers one question: **"How much power should a healthy inverter produce right now, given the
weather of the last hours?"** This is a *normal-behaviour model* (NBM): it is trained on healthy behaviour
only, so a faulty inverter moves away from the LSTM line and the gap shows the fault
(Malhotra et al., 2015; Hundman et al., 2018; NBM practice for wind turbines, Chesterman et al., 2023).

### What changed compared with the previous LSTM, and why

| Previous LSTM | Rebuilt LSTM | Why |
|---|---|---|
| Trained on **all** readings, including 2,300 training windows with 0 kW in full sun | Trained on **healthy** readings only (faulty readings removed with a robust 3σ rule against the healthy peers) | If faults are in the training targets, the model learns that "0 kW in full sun" is possible and its prediction is pulled down, so faults look smaller |
| Inputs: 3 weather features | Inputs: 3 weather features **+ time of day** (sin, cos) | The sensor and the panels are not tilted the same way; time of day tells the model where the sun is |
| Windows built on each inverter's run; they jump from evening to the next morning and readings at the start of each run get no prediction (36,491 of 53,342 training readings had one) | Windows on the **continuous 15-minute weather timeline** (night included) | Every daylight reading gets a prediction and every window is real, consecutive weather |
| One model for both plants | **One model per plant** | Different sites and sensors; the per-plant experiment of the team already showed a lower error |
| Linear output (692 negative predictions in training) | **Softplus** output | Power can never be negative |
| One network | **Average of 3 networks** (3 seeds) | Less dependence on the random start |

**The target.** The input depends only on the plant's weather, so all inverters of a plant get the *same*
input at a given moment (the previous model already predicted the same line for all 22 inverters).
Training with 22 copies of the same input and 22 targets is the same, for the MSE loss, as training once with
the **mean of the healthy inverters**, weighted by how many they are — so we do that (same optimum, 22× faster).
The target is divided by the inverter capacity (≈ 1,345 kW) so it lies between 0 and ≈ 1.

In [ ]:
import sys
from pathlib import Path

def find_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path('/content/SUNSHIELD'),
                  Path('/content/drive/MyDrive/SUNSHIELD')]
    for p in candidates:
        if (p / 'sunshield' / 'config.py').exists():
            return p
    if 'google.colab' in sys.modules:
        from google.colab import drive
        drive.mount('/content/drive')
        if (candidates[-1] / 'sunshield' / 'config.py').exists():
            return candidates[-1]
    raise FileNotFoundError('Open this notebook from the SUNSHIELD folder, or put the folder in /content '
                            'or in My Drive on Google Drive.')

ROOT = find_root()
sys.path.insert(0, str(ROOT))

In [ ]:
import os, json, time
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from IPython.display import display

from sunshield import config as C, model as M, plots as P

P.use_style()
pd.set_option('display.width', 160)
tf.config.experimental.enable_op_determinism()
IN, OUT = C.OUT_DIR / 'step1', C.OUT_DIR / 'step2'
OUT.mkdir(parents=True, exist_ok=True)

weather = pd.read_csv(IN / 'weather.csv', parse_dates=['DATE_TIME'])
readings = pd.read_csv(IN / 'readings.csv', parse_dates=['DATE_TIME'])
params = json.load(open(IN / 'params.json'))
CAPACITY = {int(p): v for p, v in params['capacity_kW'].items()}
print('TensorFlow', tf.__version__, '|', f'{len(readings):,} daylight readings')

## Step 1 — Healthy training targets

A reading is **healthy** if its deviation from the healthy peers,
$(P - P_{peer}) / \max(P_{peer},\ 20\,\%\ \text{of capacity})$, is within **3σ** of that inverter's usual value
(centre = median, σ = max(1.4826 × MAD, 10 %), computed on the training days only — the same robust
statistics as the anomaly definition, Leys et al., 2013), and it is not in a plant-wide event.
Only healthy readings are averaged into the target.
This removes faults from the training target; it does not remove timestamps (cloudy moments stay in).

In [ ]:
targets, healthy = M.healthy_targets(readings)
readings['HEALTHY'] = healthy
display(readings.groupby(['PLANT_ID', 'SPLIT'])['HEALTHY'].mean().unstack()[C.SPLITS].round(3)
        .rename_axis(columns='share of readings that are healthy'))
zero_in_sun = (readings['AC_POWER'] == 0) & (readings['IRRADIATION'] > 0.2)
print(f"Readings with 0 kW while irradiation > 0.2 kW/m²: {zero_in_sun.sum():,} — kept as healthy: "
      f"{(zero_in_sun & readings['HEALTHY']).sum()}")
display(targets.groupby(['PLANT_ID', 'SPLIT']).size().unstack()[C.SPLITS]
        .rename_axis(columns='timestamps with a target'))

## Step 2 — Window-size experiment

Same settings and seed for every window size; the window ends at the reading being predicted
(the weather of that moment is the most important input). The window with the **lowest mean validation
loss over the two plants** is kept. Inputs are standardised with the mean / std of the **training days only**.

In [ ]:
stats = M.fit_feature_scaler(weather)
timeline = M.scaled_timeline(weather, stats)

experiment, first_models = [], {}
for W in C.WINDOW_SIZES:
    for plant in C.PLANTS:
        data = M.plant_data(timeline, targets, plant, W, CAPACITY[plant])
        start = time.time()
        model, hist = M.train_one(data, W, C.SEED)
        b = int(np.argmin(hist['val_loss']))
        first_models[(W, plant)] = (model, hist)
        experiment.append({'window': W, 'hours': W / 4, 'plant': plant, 'train samples': len(data['train'][1]),
                           'epochs run': len(hist['val_loss']), 'best epoch': b + 1,
                           'val loss (MSE, scaled)': hist['val_loss'][b], 'minutes': (time.time() - start) / 60})
    print(f'window {W:2d} done')
experiment = pd.DataFrame(experiment)
summary = experiment.groupby('window')['val loss (MSE, scaled)'].mean().rename('mean val loss (2 plants)')
display(experiment.round(5))
display(summary.round(5).to_frame())
BEST_W = int(summary.idxmin())
print(f'Chosen window: {BEST_W} readings = {BEST_W / 4:g} hours (lowest mean validation loss)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(summary.index, summary.values, color=P.BLUE, marker='o', markersize=8)
ax.plot(BEST_W, summary[BEST_W], 'o', markersize=14, markerfacecolor='none', markeredgecolor=P.INK)
ax.annotate(f'lowest: W = {BEST_W}', (BEST_W, summary[BEST_W]), textcoords='offset points', xytext=(12, 6))
ax.set_xticks(C.WINDOW_SIZES)
ax.set_xlabel('Window size (readings of 15 min)')
ax.set_ylabel('Mean validation loss (MSE, scaled)')
ax.set_title('Window-size experiment (mean of both plants)')
fig.tight_layout()
fig.savefig(OUT / 'window_size_experiment.png', dpi=150)
plt.show()

## Step 3 — The network

In [ ]:
M.build_lstm(BEST_W).summary()

## Step 4 — Train the final models (3 seeds per plant)
Seed 42 is the model already trained in the experiment; seeds 43 and 44 are added.
Early stopping watches the validation loss and keeps the best epoch.

In [ ]:
models, histories = {}, {}
for plant in C.PLANTS:
    data = M.plant_data(timeline, targets, plant, BEST_W, CAPACITY[plant])
    models[plant], histories[plant] = [first_models[(BEST_W, plant)][0]], [first_models[(BEST_W, plant)][1]]
    for s in range(1, C.N_SEEDS):
        m, h = M.train_one(data, BEST_W, C.SEED + s)
        models[plant].append(m)
        histories[plant].append(h)
    for s, m in enumerate(models[plant]):
        m.save(OUT / f'lstm_plant_{plant}_seed{C.SEED + s}.keras')
    print(plant, 'best validation loss per seed:', [round(min(h['val_loss']), 5) for h in histories[plant]])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, plant in zip(axes, C.PLANTS):
    h = histories[plant][0]
    ax.plot(np.arange(1, len(h['loss']) + 1), h['loss'], color=P.BLUE, label='Training loss')
    ax.plot(np.arange(1, len(h['val_loss']) + 1), h['val_loss'], color=P.ORANGE, label='Validation loss')
    b = int(np.argmin(h['val_loss']))
    ax.axvline(b + 1, color=P.INK_2, linestyle='--', linewidth=1)
    ax.annotate(f'best epoch {b + 1}', (b + 1, h['val_loss'][b]), textcoords='offset points', xytext=(6, 14))
    ax.set_yscale('log')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE (scaled, log)')
    ax.set_title(f'Plant {plant} (seed {C.SEED})')
    ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'loss_curves.png', dpi=150)
plt.show()

## Step 5 — Predicted line for every reading (train, validation, test)

In [ ]:
readings['EXPECTED_LSTM_kW'] = M.predict_readings(models, timeline, readings, BEST_W, CAPACITY)
print('Readings without a prediction (weather gap > 1 h inside the window):',
      int(readings['EXPECTED_LSTM_kW'].isna().sum()))
print('Negative predictions:', int((readings['EXPECTED_LSTM_kW'] < 0).sum()))

## Step 6 — Error of the predicted line (kW)

- **Healthy readings**: how well the LSTM learned normal behaviour (this is the model's real accuracy).
- **All readings**: includes the faults, where a large error is *wanted* (the fault is the gap).

In [ ]:
def errors(g):
    e = (g['AC_POWER'] - g['EXPECTED_LSTM_kW']).dropna()
    eh = (g.loc[g['HEALTHY'], 'AC_POWER'] - g.loc[g['HEALTHY'], 'EXPECTED_LSTM_kW']).dropna()
    return pd.Series({'RMSE healthy (kW)': np.sqrt((eh ** 2).mean()), 'MAE healthy (kW)': eh.abs().mean(),
                      'RMSE all (kW)': np.sqrt((e ** 2).mean()), 'MAE all (kW)': e.abs().mean(),
                      'RMSE healthy % of capacity': 100 * np.sqrt((eh ** 2).mean()) / CAPACITY[g.name[0]]})

metrics = readings.groupby(['PLANT_ID', 'SPLIT']).apply(errors, include_groups=False).round(2)
metrics = metrics.reindex(pd.MultiIndex.from_product([C.PLANTS, C.SPLITS], names=['PLANT_ID', 'SPLIT']))
metrics.to_csv(OUT / 'metrics.csv')
metrics

### The predicted line on test days
Plant 1 is a normal inverter; Plant 2 shows an inverter that drops out while the LSTM line and the
healthy peers keep the expected shape.

In [ ]:
test = readings[readings['SPLIT'] == 'test']
examples = [('1IF53ai7Xc0U56Y', '2020-06-13', '2020-06-14'), ('Et9kgGMDl729KT4', '2020-06-15', '2020-06-16')]
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
for ax, (inv, d0, d1) in zip(axes, examples):
    g = test[(test['SOURCE_KEY'] == inv) & test['DATE_TIME'].between(d0, pd.Timestamp(d1) + pd.Timedelta('1D'))]
    g = g.set_index('DATE_TIME').asfreq('15min').reset_index()        # gaps (night) are not joined
    P.plot_inverter_day(ax, g, f'Plant {int(g.PLANT_ID.dropna().iloc[0])}, inverter {inv}, {d0} → {d1}')
fig.tight_layout()
fig.savefig(OUT / 'predicted_line_test_examples.png', dpi=150)
plt.show()

## Step 7 — Save

In [ ]:
readings.to_csv(OUT / 'predictions.csv', index=False)
experiment.to_csv(OUT / 'window_size_experiment.csv', index=False)
with open(OUT / 'model_info.json', 'w') as f:
    json.dump({'window': BEST_W, 'seeds': [C.SEED + s for s in range(C.N_SEEDS)], 'features': C.FEATURES,
               'scaler_train_only': {str(p): v for p, v in stats.items()}}, f, indent=2)
for f in sorted(OUT.iterdir()):
    print(f'{f.name:40s} {f.stat().st_size / 1e6:6.2f} MB')